# MS-REACT EI/GC-MS 反应网络搜索

**用途**：读取 CREST 最佳构象 `.xyz`，设置 EI/GC-MS 条件并调用 MS-REACT 进行反应网络搜索，输出网络与候选反应。

**输入要求**：CREST 生成的最佳构象 `.xyz`（首行原子数、次行注释、后续为元素符号与坐标）。

**反应条件**：EI/GC-MS（电子能量、离子化类型、温度/压力、能量分布等参数，需与仓库内 MS-REACT 接口参数名一致）。


In [ ]:
from pathlib import Path
import importlib.util

np_spec = importlib.util.find_spec("numpy")
if np_spec:
    import importlib
    np = importlib.import_module("numpy")
else:
    np = None

def read_xyz(xyz_path: Path):
    text = xyz_path.read_text(encoding="utf-8").strip().splitlines()
    if len(text) < 3:
        raise ValueError("XYZ 文件行数不足，需包含原子数、注释与坐标。")
    try:
        atom_count = int(text[0].strip())
    except ValueError as exc:
        raise ValueError("XYZ 首行必须是原子数。") from exc
    comment = text[1].strip()
    symbols = []
    coords = []
    for line in text[2:]:
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) < 4:
            raise ValueError(f"坐标行格式错误: {line}")
        symbols.append(parts[0])
        coords.append([float(parts[1]), float(parts[2]), float(parts[3])])
    if atom_count != len(symbols):
        raise ValueError(f"XYZ 原子数不匹配: 期望 {atom_count}, 实际 {len(symbols)}")
    coords_array = np.array(coords) if np is not None else coords
    structure = {
        "symbols": symbols,
        "coordinates": coords_array,
        "comment": comment,
        "charge": 0,
        "multiplicity": 1,
    }
    return structure

xyz_path = Path("path/to/crest_best.xyz")
structure = read_xyz(xyz_path)
structure


In [ ]:
import importlib
import importlib.util

msreact_spec = importlib.util.find_spec("msreact")
ms_react_spec = importlib.util.find_spec("ms_react")
if msreact_spec:
    msreact = importlib.import_module("msreact")
elif ms_react_spec:
    msreact = importlib.import_module("ms_react")
else:
    raise ImportError("未找到 msreact 或 ms_react 模块，请确认已安装并与仓库一致。")

# 根据仓库内 MS-REACT 参数名调整这些键
conditions = {
    "ionization_type": "EI",
    "electron_energy_eV": 70.0,
    "temperature_K": 523.15,
    "pressure_Pa": 101325.0,
    "energy_distribution": "gaussian",
    "energy_sigma_eV": 0.5,
    "instrument": "GC-MS",
}
conditions


In [ ]:
def run_network_search(structure, conditions):
    if hasattr(msreact, "run_network_search"):
        return msreact.run_network_search(structure=structure, conditions=conditions)
    if hasattr(msreact, "search_reaction_network"):
        return msreact.search_reaction_network(structure=structure, conditions=conditions)
    raise AttributeError("未在 msreact/ms_react 中找到网络搜索入口函数。")

result = run_network_search(structure, conditions)
result


In [ ]:
import json
from pathlib import Path
import importlib.util

nx_spec = importlib.util.find_spec("networkx")
pyvis_spec = importlib.util.find_spec("pyvis")
pandas_spec = importlib.util.find_spec("pandas")

if nx_spec:
    import importlib
    nx = importlib.import_module("networkx")
else:
    nx = None

output_dir = Path("msreact_outputs")
output_dir.mkdir(exist_ok=True)
json_path = output_dir / "reaction_network.json"
graphml_path = output_dir / "reaction_network.graphml"

if isinstance(result, dict):
    json_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")

graph = None
if isinstance(result, dict) and "graph" in result:
    graph = result["graph"]
elif hasattr(result, "graph"):
    graph = result.graph

if nx is not None and graph is not None and isinstance(graph, nx.Graph):
    nx.write_graphml(graph, graphml_path)
    if pyvis_spec:
        from pyvis.network import Network
        net = Network(height="600px", width="100%", directed=isinstance(graph, nx.DiGraph))
        net.from_nx(graph)
        html_path = output_dir / "reaction_network.html"
        net.write_html(str(html_path))
        html_path
    else:
        graphml_path
elif pandas_spec and isinstance(result, dict) and "candidates" in result:
    import pandas as pd
    df = pd.DataFrame(result["candidates"])
    df
else:
    result


## 运行方式

1. 将 `xyz_path` 替换为 CREST 最佳构象 `.xyz` 路径。
2. 根据 MS-REACT 接口文档调整 `conditions` 字段名与数值（EI/GC-MS 条件、电子能量、温度/压力、能量分布等）。
3. 运行网络搜索后，输出位于 `msreact_outputs/`：
   - `reaction_network.json`：反应网络原始输出或候选反应列表。
   - `reaction_network.graphml`：若结果包含 NetworkX 图。
   - `reaction_network.html`：若安装了 pyvis，生成可交互网络可视化。
